# Customer Segmentation Query

**Scenario:** The product team needs to segment customers based on their purchasing behavior for a new feature rollout.

**Database Schema:**
- user_activity
    - user_id
    - last_login_date
    - feature_usage_count
    - account_type

- transactions
    - transaction_id
    - user_id
    - transaction_date
    - amount
    - platform

- user_preferences
    - user_id
    - communication_preference
    - interface_theme
    - notification_settings

**Task: Create a SQL query to identify:**

- Active users (logged in last 30 days)

**Step 1**
Filter active users.
```sql 
last_login_date >= today - 30 days
```
- Filter by high-value customers (top 20% by spending)

**Step 2**

Join with transactions
```sql 
user_activity
    INNER JOIN transactions
        ON user_id
```
**Step 3**

Calculate total spending.
Instead of thinking about every transaction, we think about every customer.
i.e.,
```text
John
20
15
80
```
becomes
```text
John
115
```
using
```sql
SUM(amount)
GROUP BY user_id
```
Now we have something like:

| **user_id** | **total_spent** |
| --- | --- |
| 1 | 340 |
| 2 | 1900 |
| 3 | 85 |
| 4 | 700 |

**Step 4**

Rank customers by spending
This where the window function come in.
SQL silently adds another column.

| **user_id** | **total_spent** | **bucket** |
| --- | --- | --- |
| 1 | 340 | 1 |
| 2 | 1900 | 1 |
| 3 | 85 | 2 |
| 4 | 700 | 5 |

If we use `NTILE(5)` ordered by spending **descending** then:
* Bucket 1 = highest 20%
* Bucket 2 = next 20%
* ...
* Bucket 5 = lowest 20%
Then we simply keep:
```text
bucket = 1
```
- User preference trends for the identified customers

**Step 5**

**preference trends**

Now we join one more time:
```sql
...
JOIN user_preferences
ON user_id
```
To return something like
```text
Email
SMS
Push Notification
```

In [ ]:
# Your Prompt:
"""
Create a SQL query to find active users, high-value customers, and their preferences.
"""

# Generated Code:
"""
SELECT
    ua.user_id,
    up.communication_preference,
    up.interface_theme,
    up.notification_settings,
    SUM(t.amount) AS total_spent
FROM user_activity ua
JOIN transactions t
    ON ua.user_id = t.user_id
JOIN user_preferences up
    ON ua.user_id = up.user_id
WHERE ua.last_login_date >= CURRENT_DATE - INTERVAL 30 DAY
GROUP BY
    ua.user_id,
    up.communication_preference,
    up.interface_theme,
    up.notification_settings
ORDER BY total_spent DESC;
"""

# Follow-up Prompts:
"""
1. The query orders customers by spending, but how do I return only the top 20%?
2. Should I use a window function instead of LIMIT?
3. What's the difference between NTILE(), PERCENT_RANK(), and CUME_DIST()?
4. Should active users be filtered before or after calculating total spending?
5. Is INNER JOIN or LEFT JOIN more appropriate for this scenario?
"""

# Final Solution:
"""
WITH active_users AS (
    SELECT user_id
    FROM user_activity
    WHERE last_login_date >= CURRENT_DATE - INTERVAL 30 DAY
),
customer_spending AS (
    SELECT
        t.user_id,
        SUM(t.amount) AS total_spent
    FROM transactions t
    INNER JOIN active_users au
        ON t.user_id = au.user_id
    GROUP BY t.user_id
),
ranked_customers AS (
    SELECT
        user_id,
        total_spent,
        NTILE(5) OVER (ORDER BY total_spent DESC) AS spending_bucket
    FROM customer_spending
)
SELECT
    rc.user_id,
    rc.total_spent,
    up.communication_preference,
    up.interface_theme,
    up.notification_settings
FROM ranked_customers rc
INNER JOIN user_preferences up
    ON rc.user_id = up.user_id
WHERE rc.spending_bucket = 1
ORDER BY rc.total_spent DESC;
"""

**Disscussion Points:**
1. How were percentile calculations handled?

The requirement was to identify the **top 20% of customers by total spending.**

First, total spending was calculated for each customer by aggregating transactions with `SUM(amount)` and grouping by `user_id`.

After aggregation, a **window function** `(NTILE(5))` was used to divide customers into five equally sized groups based on total spending in descending order. Customers in the first bucket `(NTILE = 1) `represent approximately the highest-spending 20%.

Alternative approaches include:

* `PERCENT_RANK()` when working with percentile rankings.
* `CUME_DIST()` when selecting all customers above a cumulative distribution threshold.

`NTILE(5)` was chosen because it directly maps to five 20% buckets

2. What approaches to date filtering were used?

The requirement specified **active users who logged in within the last 30 days**.

The filtering was performed before aggregation to reduce the number of rows processed in later steps.

Conceptually:

```sql
last_login_date >= CURRENT_DATE - INTERVAL '30 days'
```

Filtering early has two benefits:

* Only active users participate in later joins and calculations.
* The query processes fewer records, improving efficiency.

If the business requirement changed to "top spenders in the last 30 days," an additional date filter would also be applied to the `transactions` table.

3. How was the query optimized, CTE, subquery etc…?

The solution was organized using **Common Table Expressions (CTEs)**.

Each CTE handled one logical step:

1. Filter active users.
2. Calculate customer spending.
3. Rank customers by spending.
4. Join the ranked customers with their preferences.

This modular approach improves readability and makes debugging easier because each transformation can be verified independently.

The optimization strategy also included:

* Filtering active users before joining.
* Aggregating transaction data before applying window functions.
* Using `INNER JOIN` to exclude users without transactions since they cannot qualify as high-value customers.

For very large datasets, additional optimizations could include:

* Indexes on `user_id`, `last_login_date`, and `transaction_date`.
* An index on `transactions(user_id)` to speed up joins.
* Reviewing the execution plan (`EXPLAIN` or `EXPLAIN ANALYZE`) to identify bottlenecks.
